# Milvus on Zilliz Cloud: RAG from ingestion to retrieval

**Milvus** is an **open-source vector database** (Apache 2.0) built for similarity search over very large collections of vectors. **Zilliz Cloud** is **fully managed Milvus in the cloud**, run by Zilliz, the company that created Milvus: you create a cluster in the console and connect to it with an endpoint and a token. Milvus is widely used for production RAG because:

- **Open source, with a managed option:** the same code runs against Milvus Lite (a local file), your own Milvus server or Zilliz Cloud. Only the connection `uri` changes.
- **Nothing to run yourself on Zilliz Cloud:** Zilliz handles the servers, scaling, backups and upgrades.
- **Data is saved automatically:** every chunk you add is stored in the cloud straight away. There is no `save_local()` or `dump()`, and the data survives kernel restarts.
- **Shared by many users and apps:** any notebook, API or app with the endpoint and a token sees the same data.
- **A typed schema:** each collection has fields with fixed types (primary key, text, vector, metadata), like a table in a SQL database.
- **Filter expressions inside the database:** metadata filters are SQL-like strings such as `topic == "ml"`, and they run as part of the search.
- **Extras for RAG:** partitions to keep each user's data apart, sparse vectors, and built-in **BM25 full-text search** for hybrid search.
- **LangChain support:** the `langchain-milvus` package provides `Milvus`, which works like every other LangChain vector store.

Like Astra DB and Pinecone, Zilliz Cloud runs **in the cloud**, and every add or search is a request over the network. Two practical differences from Pinecone: `Milvus` **creates the collection for you** on the first `add_documents()`, and two of LangChain's defaults (the metric, and how chunks are written) need changing, as explained below.

> **In one sentence:** Milvus is an open-source vector database with a typed schema and SQL-like filters, and Zilliz Cloud runs it for you, so your vectors are stored, searched and shared without running any servers yourself.

This notebook uses `langchain-milvus` 0.4.x (`Milvus`), which talks to Milvus through `pymilvus` 3.x (`MilvusClient`). As in `01.1.chroma_cloud.ipynb`, every step of the pipeline is a small **helper function**, so you can reuse them in other notebooks or move them to a `.py` file later.

## Milvus vs. the vector stores we've used so far

| | Milvus (Zilliz Cloud) | Pinecone | Astra DB | Chroma | FAISS |
|---|---|---|---|---|---|
| **Where it runs** | In the cloud (AWS, Google Cloud or Azure), managed for you. Also on your own servers, or in your process with Milvus Lite | In the cloud (AWS, Google Cloud or Azure), managed for you | In the cloud, managed for you | Your process, or a Chroma server | Your process |
| **Open source** | Yes: Milvus. Zilliz Cloud is the managed service | No: a cloud service only | No: a cloud service (built on open-source Cassandra) | Yes | Yes |
| **Setup** | Create a cluster, then `langchain-milvus` + endpoint + token | Create an index, then `langchain-pinecone` + API key | Create a database online, then `langchain-astradb` + endpoint + token | `chromadb` + `langchain-chroma` | `faiss-cpu` |
| **Who creates the collection / index** | `Milvus`, on the first `add_documents()` | **You**, before connecting | `AstraDBVectorStore`, the first time | `Chroma`, the first time | `FAISS.from_documents` |
| **Saving** | Automatic, in the cloud | Automatic, in the cloud | Automatic, in the cloud | Automatic with `persist_directory` | Manual: `save_local()` |
| **Search** | Approximate (`AUTOINDEX` picks the index) | Approximate (Pinecone's serverless index) | Approximate (JVector graph index) | Approximate (HNSW) | Exact (Flat) or approximate |
| **Score from `similarity_search_with_score`** | With `metric_type="COSINE"`: cosine similarity from -1 to 1, **higher = better**. With LangChain's default (L2): distance, lower = better | Cosine similarity from -1 to 1: higher = better | Similarity from 0 to 1: higher = better | Distance: lower = better | Distance: lower = better |
| **Metadata filtering** | **Expression strings** (`'topic == "ml"'`), inside the database | Dict filters, inside the database | Dict filters, inside the database | Dict filters (`where`), inside the database | After the search (`fetch_k`) |
| **Adding an existing id** | `add_documents` inserts, which **isn't guaranteed** to replace it; `upsert` overwrites it | Overwrites it (upsert) | Overwrites it (upsert) | Overwrites it (upsert) | Raises `ValueError` |
| **Several users / apps** | Yes: anyone with the endpoint and a token | Yes: anyone with an API key for the project | Yes: anyone with the endpoint and a token | Yes, with a Chroma server | No |
| **Cost** | Free plan, then pay as you go. Free (open source) on your own servers | Free Starter plan, then pay as you go | Free tier, then pay as you go | Free (open source), or Chroma Cloud | Free (open source) |
| **Best for** | Production apps that want an open-source engine with a managed option, from prototypes to billions of vectors | Production apps that want a managed vector database with very little setup | Production apps that need a managed, shared, always-on vector database | Local apps that need CRUD and filtering | Fast local search over large collections |

The retrievers and LCEL chains from the previous notebooks work unchanged: only the line that creates the vector store changes.

## The building blocks: cluster, collection, schema, index

```
Zilliz Cloud account
 │
 └── Cluster "rag-bootcamp"                   one Milvus instance, in one cloud region
      │   Public Endpoint: https://....zilliz.com
      │
      └── Database "default"                  used when you don't pass db_name
           │
           └── Collection "rag_bootcamp_docs"     created by Milvus(...) on the first add_documents()
                │   schema: pk | text | vector | $meta        ← fixed when the collection is created
                │   index on "vector": AUTOINDEX, metric COSINE
                │
                ├── entity {"pk": "ai_agents_chunk_0", "text": "AI Agents: ...", "vector": [...], "source": "...", "topic": "agents"}
                ├── entity {"pk": "ai_agents_chunk_1", ...}
                └── ...
```

`Milvus` stores each chunk as one **entity** (a row) with these fields:

| Field | Type | What it holds | Example |
|---|---|---|---|
| `pk` | `VARCHAR`, primary key | The chunk id | `"ai_agents_chunk_0"` |
| `text` | `VARCHAR` (`max_length` 65,535) | The chunk text | `"AI Agents: An Introduction ..."` |
| `vector` | `FLOAT_VECTOR` (`dim` 1536) | The embedding | 1536 numbers for `text-embedding-3-small` |
| `$meta` | `JSON`, the **dynamic field** | Every metadata key, with `enable_dynamic_field=True` | `{"source": "/.../RAG-Bootcamp/data/raw/text/ai_agents.txt", "topic": "agents"}` |

Unlike Pinecone, the text has a field of its own, so it doesn't count towards the metadata. In search results, LangChain returns `text` as `page_content` and the other fields, `pk` included, as `metadata`.

- **Cluster:** one Milvus instance in one cloud region. Its **Public Endpoint** and a **token** are all you need to connect.
- **Database:** groups collections. Every cluster has a database called `default`.
- **Collection:** like a table. Its **schema** lists the fields and their types, and it is fixed when the collection is created. `Milvus` reads the vector `dim` from the first embedding. With `enable_dynamic_field=False` (LangChain's default) it also creates one column per metadata key of the **first** chunk.
- **Dynamic field:** with `enable_dynamic_field=True`, keys that aren't in the schema go into a hidden JSON field, `$meta`. You still filter on them by name (`topic == "ml"`), and metadata keys added later are kept instead of being dropped.
- **Index:** a vector index on `vector`, with a **metric** that is fixed when the index is created. On Zilliz Cloud you use `AUTOINDEX`, and Milvus chooses and tunes the index for you.
- **Load:** Milvus only searches collections that are **loaded** into memory. `Milvus(...)` creates the index and loads the collection for you.
- **Partitions:** a collection can be split into partitions, for example one per user, like Pinecone's namespaces. `langchain-milvus` supports this with `partition_key_field=` and `partition_names=`.

## Connecting: endpoint and token

You need a cluster, its endpoint and a token, all from the Zilliz Cloud console at [cloud.zilliz.com](https://cloud.zilliz.com):

1. **Sign up:** the **Free** plan is enough for this course.
2. **Create a cluster:** choose the free cluster type and a cloud region.
3. **Endpoint:** open the cluster and copy its **Public Endpoint**. It starts with `https://`.
4. **Token:** create an **API key** in the console, or use the cluster's username and password written as `username:password`. Either one is the `token`.

Put them in your `.env` file, next to `OPENAI_API_KEY`:

```
MILVUS_CLOUD_URI=https://....zilliz.com
MILVUS_CLOUD_TOKEN=...
```

Then pass them to `Milvus` as `connection_args`:

```python
import os
from langchain_milvus import Milvus

vectorstore = Milvus(
    embedding_function=embeddings,
    collection_name="rag_bootcamp_docs",
    connection_args={"uri": os.environ["MILVUS_CLOUD_URI"], "token": os.environ["MILVUS_CLOUD_TOKEN"]},
)
```

Install the package with `uv add langchain-milvus`. It also installs `pymilvus` and Milvus Lite.

> **Why not `MILVUS_URI`?** When it is imported, `pymilvus` loads `.env` itself and reads a variable called `MILVUS_URI`. If that value has no `https://`, `import pymilvus` fails with `Illegal uri` before any of our code runs. `pymilvus` doesn't read the `MILVUS_CLOUD_*` names, so our helper can check the values and give a clear error message instead.

> **The same code runs anywhere.** Only `connection_args` changes:
>
> | Where Milvus runs | `connection_args` |
> |---|---|
> | Zilliz Cloud | `{"uri": "https://....zilliz.com", "token": "..."}` |
> | Your own Milvus server (Docker) | `{"uri": "http://localhost:19530"}` |
> | Milvus Lite: a local file, inside your process (macOS and Linux) | `{"uri": str(VECTOR_STORE_DIR / "milvus_demo.db")}` |

> **The token is a password.** Anyone who has it can read and delete the collections in your cluster. Keep it in `.env` (which git ignores) and never paste it into a notebook cell.

## Similarity metrics and scores

The **metric** is part of the vector index, and it is set when the index is created:

| `metric_type` | Score from `similarity_search_with_score` | Notes |
|---|---|---|
| `"COSINE"` | Cosine similarity from -1 to 1: **higher = better** | Works for any embeddings. **Used in this notebook** |
| `"IP"` | Inner product: higher = better | Same ranking as cosine for **normalised** vectors (OpenAI embeddings are normalised) |
| `"L2"` | Squared Euclidean distance: **lower = better** | **LangChain's default** when you don't pass `index_params` |

So we pass the index settings when we connect:

```python
INDEX_PARAMS = {"index_type": "AUTOINDEX", "metric_type": "COSINE"}

vectorstore = Milvus(..., index_params=INDEX_PARAMS)
```

`Milvus` only uses `index_params` to **create** the index, on the first `add_documents()`. After that, searches use the metric of the existing index. But LangChain also reads `index_params` to turn scores into relevance scores, and without it, it assumes L2. So pass the same `index_params` **every time you connect**.

**Relevance scores:** `similarity_search_with_relevance_scores` and `as_retriever(search_type="similarity_score_threshold")` rescale the cosine similarity to 0–1, with the same formula as Pinecone and Astra DB:

$$\text{relevance score} = \frac{1 + \text{cosine similarity}}{2}$$

| Cosine similarity (`..._with_score`) | Relevance score (`..._with_relevance_scores`) | Meaning |
|---|---|---|
| `1.0` | `1.0` | Same meaning |
| `0.4` | `0.7` | A reasonable match |
| `0` | **`0.5`** | Unrelated |
| `-1.0` | `0` | Opposite |

So a relevance threshold of `0.7` means "cosine similarity above 0.4". Because 0.5 already means "unrelated", choose a threshold well above 0.5.

## Working with Milvus in LangChain

| Method | What it does |
|---|---|
| `Milvus(embedding_function=..., collection_name=..., connection_args=..., index_params=...)` | Connect. If the collection doesn't exist, it is created on the first `add_documents()` |
| `Milvus.from_documents(docs, embedding, ids=..., collection_name=..., connection_args=...)` | Connect and add the chunks in one step |
| `vectorstore.add_documents(docs, ids=...)` | Embed and **insert** chunks. Creates the collection the first time. Doesn't check for existing ids |
| `vectorstore.upsert(ids=..., documents=...)` | Embed and **upsert** chunks: an id that already exists is **overwritten**. The collection must already exist |
| `vectorstore.similarity_search(query, k=4, expr=...)` | Return the `k` most similar `Document`s (default `k=4`) |
| `vectorstore.similarity_search_with_score(query, k=4, expr=...)` | Return `(Document, score)` pairs, with the raw scores of the index metric |
| `vectorstore.similarity_search_with_relevance_scores(query, k=4, expr=...)` | Return `(Document, score)` pairs, with scores rescaled to 0–1 |
| `vectorstore.max_marginal_relevance_search(query, k, fetch_k, expr=...)` | Return relevant **and** varied chunks (MMR) |
| `vectorstore.search_by_metadata(expr, limit=10)` | Return the chunks that match an expression, without a vector search |
| `vectorstore.delete(ids=[...])` | Remove chunks by id |
| `vectorstore.delete(expr="...")` | Remove every chunk that matches an expression |
| `vectorstore.drop()` | Delete the whole collection and its data |
| `vectorstore.as_retriever(search_kwargs={"k": 3, "expr": ...})` | Use it in a RAG chain, like the other retrievers |

Every method also has an async version: `aadd_documents`, `asimilarity_search`, `adelete`, ... The filter argument is called **`expr`**, not `filter`.

For everything else, use `pymilvus`'s `MilvusClient`: your own (`MilvusClient(uri=..., token=...)`) or the one inside the vector store (`vectorstore.client`):

| `MilvusClient` call | What it does |
|---|---|
| `client.list_collections()` | The collections in the database |
| `client.describe_collection(name)` | The schema: fields, types, `dim` and whether the dynamic field is on |
| `client.list_indexes(name)`, `client.describe_index(name, index_name)` | The index type and metric |
| `client.query(name, filter="", output_fields=["count(*)"])` | Count the chunks |
| `client.get(name, ids=[...])` | Fetch chunks by id |
| `client.drop_collection(name)` | Delete a collection and its data |

### Things to watch out for

- **`add_documents` doesn't upsert.** It sends an **insert**, and Milvus doesn't promise that inserting an id that already exists replaces the old chunk: depending on the Milvus version, you can end up with **two copies**. `upsert()` always overwrites, but only works once the collection exists. Our `ingest_chunks()` helper uses `add_documents()` to create the collection and `upsert()` after that.
- **Pass `ids` and keep `auto_id=False`.** Without ids, LangChain switches to `auto_id=True` and Milvus generates integer ids, so every run adds new chunks.
- **The first chunks fix the schema.** The vector `dim` comes from the first `add_documents()`, and so do the metadata columns unless `enable_dynamic_field=True`. Without the dynamic field, metadata keys that the first chunk didn't have are **silently dropped**. A new embedding model or dimension means a new collection.
- **Pass `index_params` every time.** It sets the metric when the collection is created, and LangChain uses it for relevance scores afterwards.
- **Filters are strings.** Write `expr='topic == "ml"'`, not `filter={"topic": "ml"}`. String values need their own quotes inside the expression.
- **Consistency.** LangChain creates the collection with `consistency_level="Session"`: your own writes show up in your own searches straight away, and other clients may see them a moment later. A query with `consistency_level="Strong"` waits for every write.
- **Every call goes over the network.** Each search embeds the question and sends a request to Zilliz Cloud. `add_documents` and `upsert` send the chunks in batches (`batch_size=1000` by default).
- **Same embedding model.** As with every vector store, query with the model that built the collection.
- **Deleting is permanent.** `delete()`, `drop()` and `drop_collection()` remove the data in the cloud, for every user of the collection.

## Filter expressions

Filters are **boolean expression strings**, close to a SQL `WHERE` clause, passed as `expr=`. They run **inside the database** as part of the search, so you get `k` results whenever at least `k` chunks match:

```python
# One topic
results = vectorstore.similarity_search("What is reinforcement learning?", k=3, expr='topic == "ml"')

# ml or dl
results = vectorstore.similarity_search("What is reinforcement learning?", k=3, expr='topic in ["ml", "dl"]')

# The same filter in a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3, "expr": 'topic == "dl"'})
```

| Operator | Meaning | Example |
|---|---|---|
| `==`, `!=` | equal / not equal | `topic == "ml"` |
| `>`, `>=`, `<`, `<=` | greater / less than | `page >= 3` |
| `in`, `not in` | in / not in a list | `topic in ["ml", "dl"]` |
| `like` | text pattern: `%` matches any characters | `source like "%/data/raw/text/%"` |
| `and`, `or`, `not` | combine conditions | `topic == "ml" or topic == "dl"` |
| `json_contains` | a list in the metadata contains a value | `json_contains(tags, "rag")` |

Metadata keys in the dynamic field are used by name, as above, or as `$meta["topic"]`. The primary key works too: `pk in ["ai_agents_chunk_0", "ai_agents_chunk_1"]`. The same expressions work with `delete(expr=...)` and `search_by_metadata(expr)`.

How this compares with the other stores:

| | Where the filter runs | Filter syntax |
|---|---|---|
| FAISS | In Python, **after** the search, on the `fetch_k` nearest chunks | Dict or Python function |
| Chroma | Inside the database | Dict (`where`) |
| Astra DB, Pinecone | Inside the database | Dict with MongoDB-style operators |
| Milvus | **Inside the database** | **Expression string** (`expr`) |

## Hybrid search

Vector search finds chunks with a **similar meaning**, but it can miss exact words such as product codes, error messages or names. Milvus handles keywords with **sparse vectors**, and it can compute them itself: a **BM25 function** in the schema turns the `text` field into a sparse vector when you insert a chunk, so you only send the text (**full-text search**).

```
question ─┬─► dense vector search    (meaning)   ─┐
          │                                      ├─► combine (RRF or weighted) ─► top k
          └─► BM25 full-text search  (keywords)  ─┘
```

In `langchain-milvus`, pass `builtin_function=BM25BuiltInFunction()` and two vector fields, `vector_field=["dense", "sparse"]`, and every search runs both and merges the results. We'll look at hybrid search in detail later in the course.

## Strengths and limits

**Strengths**

- Open source, with a managed cloud: the same code runs on Milvus Lite, your own servers or Zilliz Cloud, so you aren't tied to one provider
- Nothing to run on Zilliz Cloud: no servers, scaling or backups to handle
- Built for scale: designed for billions of vectors, with many index types (HNSW, IVF, DiskANN, GPU indexes) and `AUTOINDEX` to choose one for you
- A typed schema, with a dynamic JSON field for flexible metadata
- SQL-like filter expressions that run inside the database, during the search
- Upsert, delete by id or by expression, and fetch by id
- Partitions and partition keys keep each user's or customer's data apart
- Sparse vectors and built-in BM25 full-text search for keyword and hybrid search
- `Milvus` creates the collection and the index for you on the first add

**Limits to keep in mind**

- **More concepts to learn.** The schema, index, metric, load and consistency level all matter, where Pinecone needs one index and an API key.
- **LangChain's defaults need changing.** The default metric is L2, and `add_documents` inserts instead of upserting, so re-running it isn't guaranteed to overwrite chunks.
- **Needs an account and the internet** for Zilliz Cloud. Every call is a network request, and your app depends on a cloud service.
- **Slower per query than in-process stores.** A network round trip costs more than a FAISS or in-memory search, which matters for small local projects.
- **Costs grow with use.** Beyond the free plan you pay for compute, storage and requests, and the free plan has its own limits.
- **Fixed schema.** Changing the vector dimension, the metric or the embedding model means creating a new collection and adding the data again.
- **Self-hosting is real work.** A production Milvus cluster runs several services (such as etcd and object storage), and Milvus Lite only suits prototypes and small local projects.
- **Secrets to manage.** The token gives access to the cluster's collections, so it must stay out of your code and out of git.

## The pipeline: one helper per step

```
INGESTION  (run once, and again when the files change)

 data/raw/text/*.txt ─► load_documents() ─► add_topic_metadata() ─► split_documents() ─► make_chunk_ids()
                                                                                                │
                                                                                                ▼
                                    ingest_chunks(): embed + insert/upsert in batches ─► Zilliz Cloud collection

RETRIEVAL + GENERATION  (every question)

 question ─► get_retriever() ─► top-k chunks ─► build_rag_chain(): prompt + LLM ─► ask(): answer + sources
```

| Step | Helper | What it does |
|---|---|---|
| Connect | `get_connection_args()` | Reads `MILVUS_CLOUD_URI` and `MILVUS_CLOUD_TOKEN` from `.env` and checks them |
| | `get_milvus_client()` | Connects a `MilvusClient` and checks the endpoint and token |
| | `get_cloud_vectorstore()` | Opens the collection as a LangChain vector store, with the COSINE index settings |
| | `count_chunks()` | Counts the chunks in a collection |
| | `show_collections()` | Lists the collections in the cluster and their sizes |
| | `show_collection_info()` | Prints a collection's schema, vector index and size |
| Ingest | `load_documents()` | Loads every `.txt` file in a folder |
| | `add_topic_metadata()` | Adds a `topic` to each document, for filtering later |
| | `split_documents()` | Splits the documents into chunks |
| | `make_chunk_ids()` | Stable ids per file: `ai_agents_chunk_0`, `ai_agents_chunk_1`, ... |
| | `ingest_chunks()` | Embeds and writes the chunks in batches: insert to create the collection, upsert after that |
| | `run_ingestion()` | Runs all the ingestion steps in one call |
| Retrieve | `get_retriever()` | A retriever with `k`, an optional filter expression and an optional relevance threshold |
| | `search_with_scores()` | Prints the closest chunks with their cosine similarity or relevance score |
| Generate | `format_docs()` | Joins the chunks into the prompt context, with `[Source: ...]` labels |
| | `build_rag_chain()` | An LCEL chain that returns the answer **and** the chunks it used |
| | `ask()` | Runs the chain and prints the answer and its sources |
| Manage | `get_chunks()` | Fetches chunks by id, without a vector search |
| | `delete_chunks()` | Deletes chunks by id or by filter expression |

### Before you start

Fill in the two Milvus values in the `.env` file in the project root (the keys are already there):

```
MILVUS_CLOUD_URI=https://....zilliz.com
MILVUS_CLOUD_TOKEN=...
```

`MILVUS_CLOUD_URI` is the cluster's **Public Endpoint**, and `MILVUS_CLOUD_TOKEN` is an API key or `username:password`. `OPENAI_API_KEY` is used for the embeddings and the LLM, as in the other notebooks.

In [1]:
## Setup: shared project paths from src.config, which also loads the API keys from .env
import os
from pathlib import Path
from dotenv import load_dotenv

from src.config import PROJECT_ROOT, RAW_TEXT_DIR

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [2]:
## Load libraries
import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict

from pymilvus import MilvusClient

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_milvus import Milvus

In [3]:
## Settings used by the helpers
DATA_DIR = RAW_TEXT_DIR
COLLECTION_NAME = "rag_bootcamp_docs"   # the Milvus collection used by this notebook

EMBEDDING_MODEL = "text-embedding-3-small"   # 1536 dimensions: Milvus reads the dimension from the first chunk
LLM_MODEL = "gpt-5"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
TOP_K = 3

# The vector index: AUTOINDEX lets Milvus choose the index, COSINE gives cosine similarity scores (higher = better).
# Without it, langchain-milvus creates an L2 index (distances: lower = better).
INDEX_PARAMS = {"index_type": "AUTOINDEX", "metric_type": "COSINE"}

TOPIC_BY_FILE = {
    "ai_agents.txt": "agents",
    "machine_learning_algorithms.txt": "ml",
    "reinforcement_learning.txt": "rl",
    "deep_learning.txt": "dl",
}

## 1. Connect to Zilliz Cloud

In [4]:
## Helpers: read the connection settings, connect to Zilliz Cloud, and open a collection as a LangChain vector store
def get_connection_args():
    """Read MILVUS_CLOUD_URI and MILVUS_CLOUD_TOKEN from .env and return them as Milvus connection arguments."""
    load_dotenv(PROJECT_ROOT / ".env", override=True)   # override: picks up values filled in after the kernel started
    uri = os.getenv("MILVUS_CLOUD_URI", "").strip()
    token = os.getenv("MILVUS_CLOUD_TOKEN", "").strip()

    if not uri or not token:
        raise ValueError(
            "MILVUS_CLOUD_URI or MILVUS_CLOUD_TOKEN is empty: add the cluster's Public Endpoint and a token "
            "(an API key, or username:password) to the .env file in the project root and save the file"
        )
    if not uri.startswith("https://"):
        raise ValueError(f"MILVUS_CLOUD_URI should be the cluster's Public Endpoint, starting with https://, not: {uri}")
    return {"uri": uri, "token": token}


def get_milvus_client(connection_args):
    """Connect a MilvusClient to the cluster. list_collections() is an authenticated call: it fails here if the endpoint or token is wrong."""
    client = MilvusClient(**connection_args)
    collections = client.list_collections()
    print(f"Connected to Milvus | collections: {collections}")
    return client


def get_cloud_vectorstore(embedding, connection_args, collection_name=COLLECTION_NAME):
    """Open the collection as a LangChain vector store. If it doesn't exist yet, the first add_documents() creates it."""
    return Milvus(
        embedding_function=embedding,
        collection_name=collection_name,
        connection_args=connection_args,
        index_params=INDEX_PARAMS,   # pass it on every connect: LangChain also uses it for relevance scores
        enable_dynamic_field=True,   # keep every metadata key (in the JSON field $meta), not only the first chunk's
        auto_id=False,               # we pass our own string ids, so re-running overwrites instead of duplicating
    )


def count_chunks(client, collection_name=COLLECTION_NAME):
    """Number of chunks in the collection. consistency_level="Strong" waits for writes that are still being applied."""
    result = client.query(collection_name, filter="", output_fields=["count(*)"], consistency_level="Strong")
    return result[0]["count(*)"]


def show_collections(client):
    """Print every collection in the cluster with its number of chunks."""
    names = client.list_collections()
    print(f"{len(names)} collection(s) in the cluster:")
    for name in names:
        print(f"  - {name}: {count_chunks(client, name)} chunks")


def show_collection_info(client, collection_name=COLLECTION_NAME):
    """Print the collection's schema (fields and types), its vector index and its number of chunks."""
    info = client.describe_collection(collection_name)
    print(f"Collection '{collection_name}' | dynamic field ($meta): {info['enable_dynamic_field']}")
    print("Fields:")
    for field in info["fields"]:
        details = [f"{key}={value}" for key, value in field.get("params", {}).items()]
        if field.get("is_primary"):
            details.append("primary key")
        print(f"  - {field['name']}: {field['type'].name} {', '.join(details)}")

    for index_name in client.list_indexes(collection_name):
        index = client.describe_index(collection_name, index_name)
        print(f"Index on '{index['field_name']}': {index['index_type']}, metric {index['metric_type']}")
    print(f"Chunks: {count_chunks(client, collection_name)}")

## 2. Data ingestion

In [5]:
## Helpers: load, tag, split, give each chunk a stable id, and write the chunks to Milvus
def load_documents(doc_dir=DATA_DIR, glob="*.txt"):
    """Load every matching file in doc_dir as a LangChain Document."""
    loader = DirectoryLoader(doc_dir, glob=glob, loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    documents = loader.load()
    print(f"Loaded {len(documents)} documents from {doc_dir}")
    return documents


def add_topic_metadata(documents, topic_by_file=TOPIC_BY_FILE):
    """Add a "topic" to each document's metadata, based on its file name ("general" if the file isn't listed)."""
    for doc in documents:
        doc.metadata["topic"] = topic_by_file.get(Path(doc.metadata["source"]).name, "general")
    return documents


def split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Split the documents into chunks. Each chunk keeps the metadata of its document."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],   # try paragraphs first, then lines, then words, then characters
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Split into {len(chunks)} chunks")
    return chunks


def make_chunk_ids(chunks):
    """Stable ids per file (ai_agents_chunk_0, ai_agents_chunk_1, ...), so re-running upserts instead of duplicating."""
    position = defaultdict(int)
    ids = []
    for chunk in chunks:
        stem = Path(chunk.metadata["source"]).stem
        ids.append(f"{stem}_chunk_{position[stem]}")
        position[stem] += 1
    return ids


def ingest_chunks(vectorstore, chunks, ids, batch_size=100):
    """Embed the chunks and write them, batch_size chunks per request.

    add_documents() inserts, and creates the collection (schema + index) from the first batch.
    Once the collection exists, upsert() overwrites the chunks whose id is already there.
    """
    for start in range(0, len(chunks), batch_size):
        end = min(start + batch_size, len(chunks))
        if vectorstore.client.has_collection(vectorstore.collection_name):
            vectorstore.upsert(ids=ids[start:end], documents=chunks[start:end])
            print(f"  upserted chunks {start + 1} to {end}")
        else:
            vectorstore.add_documents(chunks[start:end], ids=ids[start:end])
            print(f"  created the collection and inserted chunks {start + 1} to {end}")
    print(f"Collection '{vectorstore.collection_name}' now has {count_chunks(vectorstore.client, vectorstore.collection_name)} chunks")


def run_ingestion(vectorstore, doc_dir=DATA_DIR):
    """Load, tag, split and write every text file in doc_dir. Returns the chunks."""
    documents = add_topic_metadata(load_documents(doc_dir))
    chunks = split_documents(documents)
    ingest_chunks(vectorstore, chunks, make_chunk_ids(chunks))
    return chunks

## 3. Retrieval

With the `COSINE` index, `similarity_search_with_score` returns the **cosine similarity**: **higher = more similar**. Relevance scores rescale it to 0–1, so unrelated chunks score about 0.5. Filters are expression strings passed as `expr`, such as `'topic == "rl"'` or `'topic in ["ml", "dl"]'`, and run **inside the database**, so you still get `k` results when at least `k` chunks match.

In [6]:
## Helpers: a retriever with an optional filter and relevance threshold, and a scored search for checking results
def get_retriever(vectorstore, k=TOP_K, expr=None, score_threshold=None):
    """A retriever that returns the k closest chunks, optionally only the ones that match a filter expression.

    With score_threshold (a relevance score from 0 to 1), k becomes a maximum: chunks below the threshold are dropped.
    """
    search_kwargs = {"k": k}
    if expr:
        search_kwargs["expr"] = expr
    if score_threshold is None:
        return vectorstore.as_retriever(search_kwargs=search_kwargs)

    search_kwargs["score_threshold"] = score_threshold
    return vectorstore.as_retriever(search_type="similarity_score_threshold", search_kwargs=search_kwargs)


def search_with_scores(vectorstore, query, k=TOP_K, expr=None, relevance=False):
    """Print the k closest chunks with their cosine similarity (-1 to 1), or with relevance=True their relevance score (0 to 1)."""
    if relevance:
        results = vectorstore.similarity_search_with_relevance_scores(query, k=k, expr=expr)
        label = "relevance"
    else:
        results = vectorstore.similarity_search_with_score(query, k=k, expr=expr)
        label = "cosine similarity"

    print(f"Query : {query}")
    print(f"Filter: {expr}\n")
    for i, (doc, score) in enumerate(results, start=1):
        print(f"--- Result {i} | {label}: {score:.4f} | id: {doc.metadata.get('pk')} | topic: {doc.metadata.get('topic')} ---")
        print(doc.page_content[:200] + "...\n")
    return results

## 4. RAG chain

In [7]:
## Helpers: format the context, build the LCEL RAG chain, and ask questions
rag_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question.
If you don't know the answer based on the context, say you don't know.
Mention the source file(s) you used.

Context:
{context}

Question: {question}

Answer:""")


def format_docs(docs):
    """Join the retrieved chunks into one string, each labelled with its source file."""
    return "\n\n".join(f"[Source: {doc.metadata['source']}]\n{doc.page_content}" for doc in docs)


def build_rag_chain(retriever, llm, prompt=rag_prompt):
    """LCEL chain: question -> {"docs": [Document, ...], "question": "...", "answer": "..."}."""
    answer_chain = (
        RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))   # Documents -> context string
        | prompt
        | llm
        | StrOutputParser()
    )
    return (
        {"docs": retriever, "question": RunnablePassthrough()}
        | RunnablePassthrough.assign(answer=answer_chain)   # adds "answer", keeps "docs" and "question"
    )


def ask(chain, question):
    """Run the chain, then print the answer and the id and metadata of the chunks it used."""
    result = chain.invoke(question)

    print(f"Question: {question}\n")
    print(f"Answer:\n{result['answer']}\n")
    print(f"Sources ({len(result['docs'])} chunks):")
    for doc in result["docs"]:
        print(f"  - {doc.metadata.get('pk')} | topic: {doc.metadata.get('topic')} | {doc.metadata.get('source')}")
    return result

## 5. Manage the collection

In [8]:
## Helpers: fetch chunks by id and delete chunks
def get_chunks(client, ids, collection_name=COLLECTION_NAME):
    """Fetch chunks by id, without a vector search. Returns a list of dicts (pk, text and metadata)."""
    return client.get(collection_name, ids=ids, output_fields=["text", "source", "topic"])


def delete_chunks(vectorstore, ids=None, expr=None):
    """Delete chunks by id, or every chunk that matches a filter expression. Permanent, for every user of the collection."""
    if not vectorstore.delete(ids=ids, expr=expr):   # returns False instead of raising when Milvus rejects the delete
        raise RuntimeError(f"Delete failed for ids={ids}, expr={expr}")
    print(f"Collection '{vectorstore.collection_name}' now has {count_chunks(vectorstore.client, vectorstore.collection_name)} chunks")

## 6. Run the pipeline

In [9]:
## Connect to Zilliz Cloud and open the collection
connection_args = get_connection_args()
milvus_client = get_milvus_client(connection_args)

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vectorstore = get_cloud_vectorstore(embeddings, connection_args)

show_collections(milvus_client)

Connected to Milvus | collections: ['rag_bootcamp_docs']
1 collection(s) in the cluster:
  - rag_bootcamp_docs: 44 chunks


In [10]:
## Ingest: load, tag, split and write every text file (only needed once, or when the files change)
chunks = run_ingestion(vectorstore)

print(f"\nFirst chunk metadata: {chunks[0].metadata}")

Loaded 4 documents from /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text
Split into 44 chunks
  upserted chunks 1 to 44
Collection 'rag_bootcamp_docs' now has 44 chunks

First chunk metadata: {'source': '/Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/ai_agents.txt', 'topic': 'agents'}


In [11]:
## The collection that Milvus(...) created: its fields, its vector index and its size
show_collection_info(milvus_client)

Collection 'rag_bootcamp_docs' | dynamic field ($meta): True
Fields:
  - text: VARCHAR max_length=65535
  - pk: VARCHAR max_length=65535, primary key
  - vector: FLOAT_VECTOR dim=1536
Index on 'vector': AUTOINDEX, metric COSINE
Chunks: 44


In [12]:
## Run the ingestion again: upsert overwrites the same ids, so the number of chunks stays the same
chunks = run_ingestion(vectorstore)

Loaded 4 documents from /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text
Split into 44 chunks
  upserted chunks 1 to 44
Collection 'rag_bootcamp_docs' now has 44 chunks


In [13]:
## Retrieval: check what the vector store returns before adding the LLM (cosine similarity: higher = better)
results = search_with_scores(vectorstore, "What is reinforcement learning?")

Query : What is reinforcement learning?
Filter: None

--- Result 1 | cosine similarity: 0.6758 | id: reinforcement_learning_chunk_0 | topic: rl ---
Reinforcement Learning: An Introduction

Reinforcement learning (RL) is a branch of machine learning in which an agent learns to make decisions by interacting with an environment. Instead of learning ...

--- Result 2 | cosine similarity: 0.6570 | id: machine_learning_algorithms_chunk_7 | topic: ml ---
Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward...

--- Result 3 | cosine similarity: 0.6266 | id: reinforcement_learning_chunk_11 | topic: rl ---
Conclusion
Reinforcement learning teaches agents to make a sequence of good decisions through trial and error. Combined with deep learning, it has produced systems that beat human champions at complex...



In [14]:
## The same search with relevance scores: (1 + cosine similarity) / 2
results = search_with_scores(vectorstore, "What is reinforcement learning?", relevance=True)

Query : What is reinforcement learning?
Filter: None

--- Result 1 | relevance: 0.8379 | id: reinforcement_learning_chunk_0 | topic: rl ---
Reinforcement Learning: An Introduction

Reinforcement learning (RL) is a branch of machine learning in which an agent learns to make decisions by interacting with an environment. Instead of learning ...

--- Result 2 | relevance: 0.8285 | id: machine_learning_algorithms_chunk_7 | topic: ml ---
Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward...

--- Result 3 | relevance: 0.8133 | id: reinforcement_learning_chunk_11 | topic: rl ---
Conclusion
Reinforcement learning teaches agents to make a sequence of good decisions through trial and error. Combined with deep learning, it has produced systems that beat human champions at complex...



In [15]:
## Retrieval with a filter expression: only chunks with topic == "ml"
results = search_with_scores(vectorstore, "What is reinforcement learning?", expr='topic == "ml"')

Query : What is reinforcement learning?
Filter: topic == "ml"

--- Result 1 | cosine similarity: 0.6570 | id: machine_learning_algorithms_chunk_7 | topic: ml ---
Reinforcement Learning
In reinforcement learning, an agent learns by interacting with an environment. It takes actions, receives rewards or penalties, and learns a strategy that maximizes total reward...

--- Result 2 | cosine similarity: 0.3492 | id: machine_learning_algorithms_chunk_1 | topic: ml ---
Supervised Learning
In supervised learning, the model is trained on labeled data, meaning each input comes with the correct output. The goal is to predict the output for new, unseen inputs. Supervised...

--- Result 3 | cosine similarity: 0.3298 | id: machine_learning_algorithms_chunk_10 | topic: ml ---
Conclusion
Machine learning algorithms are the foundation of modern AI systems. Understanding their strengths and weaknesses helps you choose the right tool for each problem, and more advanced techniq...



In [16]:
## Two topics with `in`: machine learning or deep learning chunks
results = search_with_scores(vectorstore, "How do neural networks learn?", expr='topic in ["ml", "dl"]')

Query : How do neural networks learn?
Filter: topic in ["ml", "dl"]

--- Result 1 | cosine similarity: 0.5880 | id: deep_learning_chunk_2 | topic: dl ---
A neural network is made up of neurons organized into layers: an input layer, one or more hidden layers, and an output layer. Each connection between neurons has a weight. During training, the network...

--- Result 2 | cosine similarity: 0.5109 | id: deep_learning_chunk_3 | topic: dl ---
which allows the network to learn complex relationships....

--- Result 3 | cosine similarity: 0.4618 | id: deep_learning_chunk_0 | topic: dl ---
Deep Learning: An Introduction

Deep learning is a subfield of machine learning that uses artificial neural networks with many layers to learn patterns from data. The word "deep" refers to the number ...



In [17]:
## RAG chain: retriever + prompt + LLM
llm = ChatOpenAI(model=LLM_MODEL, temperature=0)
rag_chain = build_rag_chain(get_retriever(vectorstore), llm)

result = ask(rag_chain, "What are the types of neural networks?")

Question: What are the types of neural networks?

Answer:
- Convolutional Neural Networks (CNNs)
- Recurrent Neural Networks (RNNs)
- Long Short-Term Memory networks (LSTMs)

Source: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/deep_learning.txt

Sources (3 chunks):
  - deep_learning_chunk_2 | topic: dl | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/deep_learning.txt
  - deep_learning_chunk_4 | topic: dl | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/deep_learning.txt
  - deep_learning_chunk_0 | topic: dl | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/deep_learning.txt


In [18]:
result = ask(rag_chain, "What are the core components of an AI agent?")

Question: What are the core components of an AI agent?

Answer:
Reasoning (via an LLM as the agent’s “brain”), tools (the agent’s “hands”), planning, memory, and retrieval systems such as RAG for up-to-date knowledge.

Source: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/ai_agents.txt

Sources (3 chunks):
  - ai_agents_chunk_0 | topic: agents | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/ai_agents.txt
  - ai_agents_chunk_11 | topic: agents | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/ai_agents.txt
  - ai_agents_chunk_9 | topic: agents | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/ai_agents.txt


In [19]:
## A RAG chain that only answers from deep learning chunks (topic == "dl")
dl_rag_chain = build_rag_chain(get_retriever(vectorstore, expr='topic == "dl"'), llm)

# deep_learning.txt doesn't cover reinforcement learning, so the LLM should say it doesn't know
result = ask(dl_rag_chain, "What is reinforcement learning?")

Question: What is reinforcement learning?

Answer:
I don’t know based on the provided context; the excerpts don’t define reinforcement learning.

Sources used:
- /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/deep_learning.txt

Sources (3 chunks):
  - deep_learning_chunk_0 | topic: dl | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/deep_learning.txt
  - deep_learning_chunk_8 | topic: dl | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/deep_learning.txt
  - deep_learning_chunk_3 | topic: dl | /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp/data/raw/text/deep_learning.txt


In [20]:
## RAG with a relevance threshold: only chunks with a relevance score >= 0.7 (cosine similarity >= 0.4)
threshold_retriever = get_retriever(vectorstore, k=4, score_threshold=0.7)   # k is now a maximum

# See how many chunks pass the threshold: an unrelated question should get few or none
for question in ["What is an AI agent?", "What is the capital of France?"]:
    print(f"{question} -> {len(threshold_retriever.invoke(question))} chunks")

threshold_rag_chain = build_rag_chain(threshold_retriever, llm)
result = ask(threshold_rag_chain, "What is the capital of France?")

What is an AI agent? -> 4 chunks


No relevant docs were retrieved using the relevance score threshold 0.7


What is the capital of France? -> 0 chunks


No relevant docs were retrieved using the relevance score threshold 0.7


Question: What is the capital of France?

Answer:
I don’t know based on the provided context.
Sources: None (no source files provided).

Sources (0 chunks):


In [21]:
## Fetch chunks by id, without a vector search
for row in get_chunks(milvus_client, ["ai_agents_chunk_0", "deep_learning_chunk_0"]):
    print(f"{row['pk']} | topic: {row['topic']} | {row['text'][:80]}...")

ai_agents_chunk_0 | topic: agents | AI Agents: An Introduction

An AI agent is a system that uses a large language m...
deep_learning_chunk_0 | topic: dl | Deep Learning: An Introduction

Deep learning is a subfield of machine learning ...


In [22]:
## Add a temporary chunk, find it, then delete it by id
note = Document(
    page_content="Milvus is an open-source vector database. Zilliz Cloud runs Milvus as a managed service.",
    metadata={"source": "notes", "topic": "general"},
)
ingest_chunks(vectorstore, [note], ["temp_note_0"])

results = search_with_scores(vectorstore, "Which database does Zilliz Cloud run?", k=1)

delete_chunks(vectorstore, ids=["temp_note_0"])
# Or by filter expression: delete_chunks(vectorstore, expr='source == "notes"')

  upserted chunks 1 to 1
Collection 'rag_bootcamp_docs' now has 45 chunks
Query : Which database does Zilliz Cloud run?
Filter: None

--- Result 1 | cosine similarity: 0.5922 | id: temp_note_0 | topic: general ---
Milvus is an open-source vector database. Zilliz Cloud runs Milvus as a managed service....

Collection 'rag_bootcamp_docs' now has 44 chunks


## Reconnecting later

The chunks are stored in Zilliz Cloud, so another notebook or app can skip the ingestion step and go straight to retrieval:

```python
connection_args = get_connection_args()
vectorstore = get_cloud_vectorstore(OpenAIEmbeddings(model="text-embedding-3-small"), connection_args)
rag_chain = build_rag_chain(get_retriever(vectorstore), ChatOpenAI(model="gpt-5", temperature=0))

result = ask(rag_chain, "What is deep learning?")
```

- **Same embedding model.** Always query with the model that embedded the chunks (`text-embedding-3-small` here).
- **Same `index_params`.** `get_cloud_vectorstore()` always passes them, so the relevance scores stay correct.
- **Adding files.** Put new `.txt` files in `data/raw/text` and run `run_ingestion(vectorstore)` again. The stable ids and `upsert()` mean existing chunks are overwritten, not duplicated.
- **Starting over.** `vectorstore.drop()` deletes the collection and its data in the cloud, for every app that uses it. The next `run_ingestion(vectorstore)` creates it again.

## Key takeaways

- **Milvus** is an open-source vector database, and **Zilliz Cloud** runs it as a managed service. `Milvus` from `langchain-milvus` plugs it into LangChain.
- You connect with the cluster's **Public Endpoint** and a **token** (an API key or `username:password`), kept in `.env` as `MILVUS_CLOUD_URI` and `MILVUS_CLOUD_TOKEN`. Only `connection_args` changes between Zilliz Cloud, your own server and Milvus Lite.
- `Milvus` **creates the collection** on the first `add_documents()`. Its schema (`pk`, `text`, `vector`, metadata) and the vector `dim` come from the first chunks, and `enable_dynamic_field=True` keeps every metadata key in `$meta`.
- Pass **`index_params={"index_type": "AUTOINDEX", "metric_type": "COSINE"}`** every time you connect: LangChain's default metric is L2.
- `similarity_search_with_score` then returns the **cosine similarity (-1 to 1, higher = better)**. Relevance scores rescale it to 0–1, so unrelated chunks score about **0.5**.
- `add_documents` **inserts**, which isn't guaranteed to replace a chunk with the same id. Once the collection exists, use **`upsert()`** with fixed ids to overwrite chunks instead of duplicating them.
- **Filters are expression strings** passed as `expr=`, such as `'topic == "ml"'` or `'topic in ["ml", "dl"]'`, and they run **inside the database**.
- `MilvusClient` (or `vectorstore.client`) covers the rest: the schema, the index, counts, fetching by id and dropping collections.
- `as_retriever()` plugs Milvus into the same LCEL chains we built with Chroma, FAISS, `InMemoryVectorStore`, Astra DB and Pinecone.